# Aperture -- Attention Rollout (Colab GPU worker)
OpenVLA-7B in 4-bit on a free Colab T4, attention rollout over self-attention,
plus a worker loop that drains the Aperture API's queued attribution jobs.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU -- Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0)
print(torch.cuda.get_device_name(0), f"{p.total_memory/1e9:.1f} GB")

### Install pinned deps
OpenVLA's `trust_remote_code` modeling was written against this exact stack; newer `transformers` frequently throws on its custom code.

In [ ]:
!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.16" \
    accelerate bitsandbytes pillow requests numpy boto3

### HF auth (optional)
`openvla/openvla-7b` is public; run only if you hit a 401/gated error.

In [ ]:
# from huggingface_hub import login; login()  # paste token when prompted

### Config

In [ ]:
import os
API_BASE = os.environ.get('APERTURE_API_BASE', 'https://aperture-api.onrender.com')
API_KEY  = os.environ.get('APERTURE_API_KEY', 'demo-key')
MODEL_ID = os.environ.get('APERTURE_VLA_MODEL', 'openvla/openvla-7b')

### Load model (4-bit + eager attention)

In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb,        # ~5 GB instead of ~15 GB; fits a free T4/P100
    attn_implementation="eager",    # REQUIRED -- flash/SDPA return attentions=None
    output_attentions=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={"": 0},
).eval()
print("loaded")

### Rollout (unchanged patent-precise method)

In [ ]:
import numpy as np, torch

def attention_rollout(attentions):
    """attentions: list of [batch, heads, tokens, tokens] per layer -> query->patch map."""
    result = None
    for attn in attentions:
        a = attn.mean(dim=1)[0]
        a = 0.5 * a + 0.5 * torch.eye(a.size(0), device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        result = a if result is None else a @ result
    q = result[0, 1:]
    g = int(np.sqrt(q.shape[0]))
    return q[: g * g].reshape(g, g).detach().cpu().numpy()

### Smoke test on one frame

In [ ]:
import requests, torch
from PIL import Image

img = Image.open(requests.get(
    "http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw).convert("RGB")
prompt = "In: What action should the robot take to pick up the object?\nOut:"

inputs = processor(prompt, img).to(0, dtype=torch.float16)
with torch.no_grad():
    out = model(**inputs, output_attentions=True)

assert out.attentions is not None, "attentions is None -- attn_implementation must be 'eager'"
heat = attention_rollout(out.attentions)
print("attn layers:", len(out.attentions), "| heatmap grid:", heat.shape)

import matplotlib.pyplot as plt
plt.imshow(heat, cmap="inferno"); plt.title("attention rollout"); plt.colorbar(); plt.show()

### R2 + API clients (only if you have credentials)

In [ ]:
import os, boto3
r2 = boto3.client('s3', endpoint_url=os.environ['APERTURE_R2_ENDPOINT_URL'],
                  aws_access_key_id=os.environ['APERTURE_R2_ACCESS_KEY_ID'],
                  aws_secret_access_key=os.environ['APERTURE_R2_SECRET_ACCESS_KEY'])
BUCKET = os.environ.get('APERTURE_R2_BUCKET', 'aperture-blobs')
HDRS = {'X-API-Key': API_KEY}

### Episode processing + poll loop

Claims queued jobs, runs the rollout on this GPU, writes each heatmap to R2, and reports the
uri back. `POST /v1/attribution_jobs/{id}/done` is what publishes the result onto the episode's
attribution, so it reaches `GET /v1/episodes/{id}/attribution` and the dashboard.

In [ ]:
import base64, io, json, time, requests, numpy as np, torch
from PIL import Image


def blob_key(uri):
    """Storage key out of a backend-native uri, mirroring `aperture.core.storage.key_from_uri`."""
    if uri.startswith('local://'):
        return uri[len('local://'):]
    if uri.startswith('r2://'):
        return uri[len('r2://'):].split('/', 1)[1]
    return uri


def episode_frames(job, max_frames=8):
    """Frame images for a job, decoded from the episode's raw uploaded blob.

    `GET /v1/episodes/{id}` returns per-frame *signals*, not pixels, so the worker goes back to
    the file the episode was ingested from — served through `/v1/blobs/{key}` by either storage
    backend. Handles both accepted shapes: RLDS `steps[].observation.image` and LeRobot
    `frames[]["observation.image"]`, each base64, with a `data:` prefix tolerated.
    """
    uri = job.get('rlds_uri')
    if not uri:
        return []
    raw = requests.get(f"{API_BASE}/v1/blobs/{blob_key(uri)}", headers=HDRS, timeout=60).content
    doc = json.loads(raw)
    rows = doc.get('steps') or doc.get('frames') or []

    images = []
    for row in rows:
        b64 = (row.get('observation') or {}).get('image') or row.get('observation.image')
        if not b64:
            continue
        images.append(Image.open(io.BytesIO(base64.b64decode(b64.split(',', 1)[-1]))))
        if len(images) >= max_frames:
            break
    return images


def attention_document(frames, instruction):
    """One row-normalized grid per frame — the same JSON shape the API's in-process path emits,
    so the dashboard renders a worker result and a local result identically."""
    seq = []
    prompt = f"In: What action should the robot take to {instruction}?\nOut:"
    for t, img in enumerate(frames):
        inputs = processor(prompt, img.convert("RGB")).to(0, dtype=torch.float16)
        with torch.no_grad():
            out = model(**inputs, output_attentions=True)
        heat = np.asarray(attention_rollout(out.attentions), dtype=np.float64)
        total = heat.sum()
        if total > 0:
            heat = heat / total
        fy, fx = np.unravel_index(int(heat.argmax()), heat.shape)
        seq.append({'t': t, 'grid': np.round(heat, 5).tolist(), 'focus': [float(fx), float(fy)]})
    return {
        'simulated': False,
        'method': 'attention_rollout',
        'grid_size': len(seq[0]['grid']) if seq else 0,
        'frames': seq,
    }


def poll_once():
    """Drain the queued jobs this API key can see. Returns how many were completed."""
    jobs = requests.get(f"{API_BASE}/v1/attribution_jobs", params={'status': 'queued'},
                        headers=HDRS, timeout=30).json()
    completed = 0
    for job in jobs:
        # 409 means another worker already claimed it -- skip rather than duplicate GPU time.
        if requests.post(f"{API_BASE}/v1/attribution_jobs/{job['id']}/claim",
                         headers=HDRS, timeout=30).status_code == 409:
            continue
        try:
            frames = episode_frames(job)
            if not frames:
                raise RuntimeError('episode carries no frame images')
            document = attention_document(frames, job.get('instruction') or 'complete the task')

            key = f"rollouts/{job['id']}.json"
            r2.put_object(Bucket=BUCKET, Key=key, Body=json.dumps(document).encode())
            requests.post(f"{API_BASE}/v1/attribution_jobs/{job['id']}/done", headers=HDRS,
                          json={'attention_map_uri': f"r2://{BUCKET}/{key}"},
                          timeout=30).raise_for_status()
            completed += 1
        except Exception as e:
            # Report the failure instead of leaving the job 'running' forever.
            requests.post(f"{API_BASE}/v1/attribution_jobs/{job['id']}/failed",
                          headers=HDRS, json={'error': str(e)[:500]}, timeout=30)
    return completed


print('Worker ready. Uncomment the loop below to start draining the queue.')
# while True:
#     print('completed', poll_once()); time.sleep(10)